# 00a — Philological seeds

**Phase 0, part 2** of the notebook chain (plan §1.5 + §2.7).

Loads the centralized normalization seeds from `data/seeds/` and verifies that
the canonical 7-step pipeline (`apps.backend.normalize.normalize_canonical`)
behaves correctly on hand-crafted classical-Chinese fixtures. Also exercises
the 紀年 → CE converter against canonical Tang reign-period samples.

Pipeline steps (per plan §2.7):

1. NFC (always)
2. whitespace collapse (always)
3. T-S unification (always; OpenCC `s2t`)
4. 異體字 / variant chars (always; `data/seeds/variants_unihan.tsv`)
5. 避諱 / dynasty taboo (era-conditional; `data/seeds/taboo_{tang,song,ming,qing}.yaml`)
6. 通假字 / phonetic loans (off by default; `data/seeds/loan_chars.tsv`)
7. mojimoji (ja/mixed only)

Inputs:

- `notebooks/_artifacts/00_setup_smoke_test/health.json` — must exist and report
  `silra.ok && neo4j.ok && minio.ok`.

Outputs:

- `notebooks/_artifacts/00a_philological_seeds/seeds.json` — counts + sha256
  checksums of every seed file + sample normalizations + era-conversion table.
  Subsequent notebooks (06 Translation Agent, 09b Verifier) consume this.

In [19]:
from __future__ import annotations

import hashlib
import json
import logging
import sys
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / 'pyproject.toml').exists(), f'cannot locate repo root from {Path.cwd()}'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / '.env')
logging.basicConfig(level=logging.WARNING, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

SEED_DIR = REPO_ROOT / 'data' / 'seeds'
ARTIFACT_DIR = REPO_ROOT / 'notebooks' / '_artifacts' / '00a_philological_seeds'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
PRIOR_ARTIFACT = REPO_ROOT / 'notebooks' / '_artifacts' / '00_setup_smoke_test' / 'health.json'

print(f'repo root      : {REPO_ROOT}')
print(f'seed dir       : {SEED_DIR}')
print(f'artifact dir   : {ARTIFACT_DIR}')
print(f'prior artifact : {PRIOR_ARTIFACT}')

repo root      : /Users/mohasani/Ancient
seed dir       : /Users/mohasani/Ancient/data/seeds
artifact dir   : /Users/mohasani/Ancient/notebooks/_artifacts/00a_philological_seeds
prior artifact : /Users/mohasani/Ancient/notebooks/_artifacts/00_setup_smoke_test/health.json


## Refuse to advance if Phase 0 health is red

In [20]:
assert PRIOR_ARTIFACT.exists(), (
    'run notebooks/00_setup_smoke_test.ipynb first to generate '
    f'{PRIOR_ARTIFACT}'
)
prior = json.loads(PRIOR_ARTIFACT.read_text(encoding='utf-8'))
for service in ('silra', 'neo4j', 'minio'):
    block = prior.get(service, {})
    print(f'  {service:<6}: ok={block.get("ok")} errors={block.get("errors", [])}')
    assert block.get('ok'), f'{service} probe was red in 00 — fix it before running 00a'
print('\nPhase 0 health is green — proceeding with seed verification.')

  silra : ok=True errors=[]
  neo4j : ok=True errors=[]
  minio : ok=True errors=[]

Phase 0 health is green — proceeding with seed verification.


## Inventory + checksum every seed file

Checksums let downstream notebooks detect when a seed has been edited and
trigger re-runs (the `bench` partition will compare these against a frozen
manifest).

In [21]:
SEED_FILES = [
    'variants_unihan.tsv',
    'taboo_tang.yaml',
    'taboo_song.yaml',
    'taboo_ming.yaml',
    'taboo_qing.yaml',
    'loan_chars.tsv',
    'era_calendar.yaml',
]

seed_inventory: dict[str, dict] = {}
for name in SEED_FILES:
    path = SEED_DIR / name
    assert path.exists(), f'missing seed file: {path}'
    raw = path.read_bytes()
    seed_inventory[name] = {
        'path': str(path.relative_to(REPO_ROOT)),
        'bytes': len(raw),
        'sha256': hashlib.sha256(raw).hexdigest(),
    }
    print(f'  {name:<24s} {len(raw):>6d} bytes  sha256={seed_inventory[name]["sha256"][:12]}...')

  variants_unihan.tsv        2800 bytes  sha256=0428aa3e1e90...
  taboo_tang.yaml            4710 bytes  sha256=952c6c907e86...
  taboo_song.yaml            1295 bytes  sha256=43a38517f1ef...
  taboo_ming.yaml            1051 bytes  sha256=c145b815ff9e...
  taboo_qing.yaml            1995 bytes  sha256=d389cfaaf775...
  loan_chars.tsv             1895 bytes  sha256=26cb0adc33d1...
  era_calendar.yaml          9879 bytes  sha256=94e91c354a33...


## Load each seed via its production loader

These imports exercise the cached loaders in `apps/backend/normalize/`. Any
schema drift in the YAML / TSV files will surface here.

In [22]:
from apps.backend.normalize import variants, taboo, loan, era as era_mod

variants_map = variants.load_map()
loan_map = loan.load_map()
loan_map_with_bi = loan.load_map(include_bi=True)
era_calendar = era_mod.load_calendar()

tables_summary = {
    'variants_pairs': len(variants_map),
    'loan_pairs_directed': len(loan_map),
    'loan_pairs_with_bi': len(loan_map_with_bi),
    'taboo_per_dynasty': {},
    'eras_per_dynasty': {},
}

for dyn in ('tang', 'song', 'ming', 'qing'):
    table = taboo.load_table(dyn)
    safe_map = taboo.build_substitution_map(dyn, unsafe=False)
    full_map = taboo.build_substitution_map(dyn, unsafe=True)
    tables_summary['taboo_per_dynasty'][dyn] = {
        'dynasty_zh': table.get('dynasty'),
        'years': table.get('years'),
        'emperors': len(table.get('emperors', [])),
        'safe_substitutions': len(safe_map),
        'all_substitutions': len(full_map),
    }

for dyn_block in era_calendar.get('dynasties', []):
    name = dyn_block['name']
    eras = dyn_block.get('eras', [])
    if eras:
        tables_summary['eras_per_dynasty'][name] = {
            'count': len(eras),
            'first': eras[0]['era_name'] + f" ({eras[0]['start_year']})",
            'last': eras[-1]['era_name'] + f" ({eras[-1]['end_year']})",
        }

print(json.dumps(tables_summary, ensure_ascii=False, indent=2))

{
  "variants_pairs": 43,
  "loan_pairs_directed": 24,
  "loan_pairs_with_bi": 28,
  "taboo_per_dynasty": {
    "tang": {
      "dynasty_zh": "唐",
      "years": [
        618,
        907
      ],
      "emperors": 16,
      "safe_substitutions": 12,
      "all_substitutions": 22
    },
    "song": {
      "dynasty_zh": "宋",
      "years": [
        960,
        1279
      ],
      "emperors": 4,
      "safe_substitutions": 5,
      "all_substitutions": 6
    },
    "ming": {
      "dynasty_zh": "明",
      "years": [
        1368,
        1644
      ],
      "emperors": 3,
      "safe_substitutions": 4,
      "all_substitutions": 4
    },
    "qing": {
      "dynasty_zh": "清",
      "years": [
        1644,
        1912
      ],
      "emperors": 5,
      "safe_substitutions": 7,
      "all_substitutions": 8
    }
  },
  "eras_per_dynasty": {
    "隋": {
      "count": 1,
      "first": "大業 (605)",
      "last": "大業 (618)"
    },
    "唐": {
      "count": 75,
      "first": "武德 (618)",

## Sample normalizations — one per table

Each row exercises exactly one transformation in isolation, so a regression
in any single normalize module is easy to localize.

In [23]:
from apps.backend.normalize import nfc, whitespace, tsc, kana

samples = []

raw = '\u00e9'  # 'é' as composed (NFC)
decomposed = 'e\u0301'  # 'é' as decomposed
samples.append({'step': 'nfc', 'input': decomposed, 'output': nfc.normalize(decomposed),
                'expected': raw, 'note': 'decomposed -> composed Unicode'})

samples.append({'step': 'whitespace', 'input': '貞觀\u3000十九\u00a0年   ',
                'output': whitespace.normalize('貞觀\u3000十九\u00a0年   '),
                'expected': '貞觀 十九 年',
                'note': 'full-width + nbsp + trailing collapse'})

samples.append({'step': 'tsc.s2t', 'input': '万岁',
                'output': tsc.normalize('万岁', config='s2t'),
                'expected_contains': '萬',
                'note': 'OpenCC s2t -> traditional'})

samples.append({'step': 'variants', 'input': '麼麽体学说',
                'output': variants.normalize('麼麽体学说'),
                'note': '異體字 unification (Phase-0 hand seed)'})

samples.append({'step': 'taboo.tang.safe', 'input': '飲泉而食',
                'output': taboo.normalize('飲泉而食', era='Tang'),
                'expected': '飲淵而食',
                'note': '泉 (safe) -> 淵 under Tang taboo'})

samples.append({'step': 'taboo.tang.unsafe', 'input': '深淵之水',
                'output': taboo.normalize('深淵之水', era='Tang', unsafe=True),
                'expected': '淵淵之水',
                'note': '深 only canonicalizes when unsafe=True (high-collision)'})

samples.append({'step': 'taboo.tang.no_era', 'input': '深淵之水',
                'output': taboo.normalize('深淵之水', era=None),
                'expected': '深淵之水',
                'note': 'no era -> no-op'})

samples.append({'step': 'loan.off_default', 'input': '早起反身',
                'output': loan.normalize('早起反身'),
                'note': '早 / 反 are 通假字 in the seed (a>b: 早->蚤, 反->返)'})

samples.append({'step': 'kana.zh_noop', 'input': 'ﾊﾝｶｸ',
                'output': kana.normalize('ﾊﾝｶｸ', lang='zh'),
                'expected': 'ﾊﾝｶｸ',
                'note': 'kana step is no-op when lang=zh*'})

samples.append({'step': 'kana.ja_normalize', 'input': 'ﾊﾝｶｸ',
                'output': kana.normalize('ﾊﾝｶｸ', lang='ja'),
                'expected': 'ハンカク',
                'note': 'kana half->full when lang=ja*'})

for s in samples:
    print(f"  {s['step']:<22s} {s['input']!r:>16s} -> {s['output']!r}")
    if 'expected' in s and s['output'] != s['expected']:
        print(f'        EXPECTED {s["expected"]!r}')
    if 'expected_contains' in s and s['expected_contains'] not in s['output']:
        print(f'        EXPECTED to contain {s["expected_contains"]!r}')

  nfc                                'é' -> 'é'
  whitespace             '貞觀\u3000十九\xa0年   ' -> '貞觀 十九 年'
  tsc.s2t                            '万岁' -> '萬歲'
  variants                        '麼麽体学说' -> '么么體學說'
  taboo.tang.safe                  '飲泉而食' -> '飲淵而食'
  taboo.tang.unsafe                '深淵之水' -> '淵淵之水'
  taboo.tang.no_era                '深淵之水' -> '深淵之水'
  loan.off_default                 '早起反身' -> '早起返身'
  kana.zh_noop                     'ﾊﾝｶｸ' -> 'ﾊﾝｶｸ'
  kana.ja_normalize                'ﾊﾝｶｸ' -> 'ハンカク'


## 紀年 → CE converter spot-checks

Canonical Tang reign-period anchors. The 干支 cross-check on the last row
intentionally injects a wrong sexagenary token to verify that `confidence`
drops below 1.0.

In [24]:
ERA_FIXTURES = [
    ('武德元年',   618),
    ('貞觀十九年', 645),
    ('開元二十年', 732),
    ('天寶十四年', 755),
    ('貞元元年',   785),
    ('元和九年',   814),
    ('大業十四年', 618),  # Sui — last year before Tang
    ('太平興國四年', 979),  # Northern Song multi-char era
]

era_results = []
for fixture, expected in ERA_FIXTURES:
    matches = era_mod.find_eras(fixture)
    actual = matches[0].ce_year if matches else None
    era_results.append({
        'fixture': fixture,
        'expected_ce': expected,
        'actual_ce': actual,
        'dynasty': matches[0].candidates[0].dynasty if matches else None,
        'emperor': matches[0].candidates[0].emperor if matches else None,
        'confidence': matches[0].confidence if matches else 0.0,
    })
    flag = '✓' if actual == expected else '✗'
    print(f'  {flag} {fixture:<10s} expected={expected}  got={actual}  '
          f'conf={era_results[-1]["confidence"]:.2f}')

# Cross-check: 開元二十年 = 732 = 壬申; deliberately give 甲申 to detect mismatch.
wrong_gz = era_mod.find_eras('開元二十年甲申')
wrong_gz_summary = {
    'fixture': '開元二十年甲申 (deliberate mismatch)',
    'ce_year': wrong_gz[0].ce_year if wrong_gz else None,
    'gz_in_text': wrong_gz[0].ganzhi_in_text if wrong_gz else None,
    'gz_expected': wrong_gz[0].ganzhi_expected if wrong_gz else None,
    'confidence': wrong_gz[0].confidence if wrong_gz else 0.0,
}
print('\n  ganzhi mismatch behavior:', wrong_gz_summary)

  ✓ 武德元年       expected=618  got=618  conf=1.00
  ✓ 貞觀十九年      expected=645  got=645  conf=1.00
  ✓ 開元二十年      expected=732  got=732  conf=1.00
  ✓ 天寶十四年      expected=755  got=755  conf=1.00
  ✓ 貞元元年       expected=785  got=785  conf=1.00
  ✓ 元和九年       expected=814  got=814  conf=1.00
  ✓ 大業十四年      expected=618  got=618  conf=1.00
  ✓ 太平興國四年     expected=979  got=979  conf=1.00

  ganzhi mismatch behavior: {'fixture': '開元二十年甲申 (deliberate mismatch)', 'ce_year': 732, 'gz_in_text': '甲申', 'gz_expected': '壬申', 'confidence': 0.6}


## Full canonical pipeline on a Tang fixture

Tests all 7 steps end-to-end. Per §2.7 defaults the loan step is OFF; we run
with it ON below as well so the output trace is visible for the ADR.

In [25]:
from apps.backend.normalize import normalize_canonical

FIXTURE = '貞觀\u300010月，唐徵高麗，飲泉之水。麼為著名 of 体学说.   '

def render(result):
    print(f'  original  : {result.original!r}')
    print(f'  canonical : {result.canonical!r}')
    print(f'  flags     : {result.flags}')
    for step in result.steps:
        marker = '*' if step.changed else ' '
        print(f'   {marker} {step.name:<10s} {step.before!r:>50s} -> {step.after!r}')

print('--- defaults (era=Tang, no loan, no kana) ---')
default_result = normalize_canonical(FIXTURE, lang='zh', era='Tang')
render(default_result)

print('\n--- with loan step ON (translator-style) ---')
with_loan_result = normalize_canonical(FIXTURE, lang='zh', era='Tang', apply_loan=True)
render(with_loan_result)

print('\n--- mixed-script (kanbun fragment) ---')
kanbun = '唐令ﾆ依ル所ノ﻿ 通典 ﾊ唐徵高麗ﾉ事'
kanbun_result = normalize_canonical(kanbun, lang='ja-kanbun', era='Tang')
render(kanbun_result)

--- defaults (era=Tang, no loan, no kana) ---
  original  : '貞觀\u300010月，唐徵高麗，飲泉之水。麼為著名 of 体学说.   '
  canonical : '貞觀 10月，唐徵高麗，飲淵之水。么為着名 of 體學說.'
  flags     : {'nfc': True, 'whitespace': True, 'tsc': True, 'variants': True, 'taboo': True, 'loan': False, 'kana': False, 'taboo_unsafe': False}
     nfc                   '貞觀\u300010月，唐徵高麗，飲泉之水。麼為著名 of 体学说.   ' -> '貞觀\u300010月，唐徵高麗，飲泉之水。麼為著名 of 体学说.   '
   * whitespace            '貞觀\u300010月，唐徵高麗，飲泉之水。麼為著名 of 体学说.   ' -> '貞觀 10月，唐徵高麗，飲泉之水。麼為著名 of 体学说.'
   * tsc                           '貞觀 10月，唐徵高麗，飲泉之水。麼為著名 of 体学说.' -> '貞觀 10月，唐徵高麗，飲泉之水。麼為著名 of 體學說.'
   * variants                      '貞觀 10月，唐徵高麗，飲泉之水。麼為著名 of 體學說.' -> '貞觀 10月，唐徵高麗，飲泉之水。么為着名 of 體學說.'
   * taboo                         '貞觀 10月，唐徵高麗，飲泉之水。么為着名 of 體學說.' -> '貞觀 10月，唐徵高麗，飲淵之水。么為着名 of 體學說.'
     loan                          '貞觀 10月，唐徵高麗，飲淵之水。么為着名 of 體學說.' -> '貞觀 10月，唐徵高麗，飲淵之水。么為着名 of 體學說.'
     kana                          '貞觀 10月，唐徵高麗，飲淵之水。么為着名 of 體學說.' -> '貞觀 10月，唐徵高麗，飲淵之水

## Persist the artifact

In [26]:
artifact = {
    'stage': '00a_philological_seeds',
    'ts': datetime.now(timezone.utc).isoformat(),
    'prior_artifact_ts': prior.get('ts'),
    'seed_inventory': seed_inventory,
    'tables_summary': tables_summary,
    'samples': samples,
    'era_results': era_results,
    'era_ganzhi_check': wrong_gz_summary,
    'pipeline_default': {
        'fixture': FIXTURE,
        'canonical': default_result.canonical,
        'flags': default_result.flags,
        'changes': [
            {'step': s.name, 'before': s.before, 'after': s.after}
            for s in default_result.steps if s.changed
        ],
    },
    'pipeline_with_loan': {
        'canonical': with_loan_result.canonical,
        'changes': [
            {'step': s.name, 'before': s.before, 'after': s.after}
            for s in with_loan_result.steps if s.changed
        ],
    },
    'pipeline_kanbun': {
        'fixture': kanbun,
        'canonical': kanbun_result.canonical,
        'flags': kanbun_result.flags,
    },
}

out = ARTIFACT_DIR / 'seeds.json'
out.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'wrote {out} ({out.stat().st_size} bytes)')

wrote /Users/mohasani/Ancient/notebooks/_artifacts/00a_philological_seeds/seeds.json (9236 bytes)


## Assertions — refuse to advance with a bad seed set

In [27]:
assert tables_summary['variants_pairs'] >= 20, 'expected >= 20 variant pairs in seed'
assert tables_summary['loan_pairs_directed'] >= 15, 'expected >= 15 directed loan pairs'
assert tables_summary['taboo_per_dynasty']['tang']['emperors'] >= 10, 'Tang taboo seed too small'
assert '唐' in tables_summary['eras_per_dynasty'], 'Tang era list missing'
assert tables_summary['eras_per_dynasty']['唐']['count'] >= 70, 'Tang era list incomplete'

for row in era_results:
    assert row['actual_ce'] == row['expected_ce'], (
        f'{row["fixture"]} expected CE {row["expected_ce"]}, got {row["actual_ce"]}'
    )

assert wrong_gz_summary['confidence'] < 1.0, '干支 mismatch should drop confidence'

for s in samples:
    if 'expected' in s:
        assert s['output'] == s['expected'], (
            f"{s['step']}: input={s['input']!r} got={s['output']!r} expected={s['expected']!r}"
        )
    if 'expected_contains' in s:
        assert s['expected_contains'] in s['output'], (
            f"{s['step']}: output {s['output']!r} missing {s['expected_contains']!r}"
        )

# Targeted fixtures for taboo-safety semantics.
deep_default = normalize_canonical('深淵之水', lang='zh', era='Tang')
assert '深' in deep_default.canonical, (
    '深 (high-collision; safe=false) must NOT canonicalize in the default pipeline; '
    f'got canonical={deep_default.canonical!r}'
)
deep_unsafe = normalize_canonical('深淵之水', lang='zh', era='Tang', taboo_unsafe=True)
assert '深' not in deep_unsafe.canonical, (
    '深 should canonicalize when taboo_unsafe=True; '
    f'got canonical={deep_unsafe.canonical!r}'
)
spring_default = normalize_canonical('飲泉而食', lang='zh', era='Tang')
assert '淵' in spring_default.canonical and '泉' not in spring_default.canonical, (
    '泉 (safe) must canonicalize to 淵 under Tang taboo; '
    f'got canonical={spring_default.canonical!r}'
)

# Loan-on variant must apply directed (a>b) pairs (e.g. 反 -> 返).
# 早/蚤 is `bi` in the seed (symmetric) so it is intentionally excluded by
# default; that's why we test 反 here.
loan_only = normalize_canonical('反身而誠', lang='zh', apply_loan=True)
assert '返' in loan_only.canonical and '反' not in loan_only.canonical, (
    f'loan step did not rewrite 反 -> 返 when apply_loan=True; got canonical={loan_only.canonical!r}'
)

print('all assertions ok — Phase 0 philological foundation is healthy.')
print('next: 01_ingestion.ipynb')

all assertions ok — Phase 0 philological foundation is healthy.
next: 01_ingestion.ipynb
